# AI 620: Fundamentals of Data Engineering
# Assignment 2 Part 2 - Web Scraping Fundamentals
### Website: http://books.toscrape.com/
**Total Points: 100**

---

## Assignment Overview

This hands-on assignment focuses exclusively on web scraping using Python. You will build progressively sophisticated scrapers for `http://books.toscrape.com/`, learning essential techniques for data extraction from static websites.

### Learning Objectives
- Send HTTP requests and parse HTML responses
- Extract structured data using BeautifulSoup
- Navigate pagination systems
- Handle errors and edge cases gracefully
- Implement ethical and polite scraping practices
- Save scraped data to CSV files


RUN THESE CELLS (DO NOT MODIFY)

In [1]:
# Run this cell to install required packages (if needed)
!pip install requests beautifulsoup4 pandas lxml

In [2]:
# Import libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from urllib.parse import urljoin
import re
from datetime import datetime

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

print("Libraries imported successfully!")

Libraries imported successfully!


# Part 1: Basic Scraping (30 points)

## Task 1.1: Explore the Website (5 points)

**Instructions:**
1. Send a GET request to `http://books.toscrape.com/`
2. Check the status code and print it
3. Parse the HTML using BeautifulSoup
4. Print the title of the page

*Hint: A successful request returns status code 200.*


In [3]:
url = 'http://books.toscrape.com/'
response = requests.get(url)
print(f"Status code: {response.status_code}")

soup = BeautifulSoup(response.content, 'html.parser')
page_title = soup.find('title').get_text(strip=True)
print(f"Page Title: {page_title}")
print("-" * 50)

Status code: 200
Page Title: All products | Books to Scrape - Sandbox
--------------------------------------------------


## Task 1.2: Extract Book Information from Homepage (15 points)

**Instructions:**

Extract the following information for **all books** displayed on the first page:

- **Title** (from `h3 > a` tag's `title` attribute)
- **Price** (convert to float, remove '£' symbol)
- **Availability** (extract text from the "In stock" element)
- **Star rating** (convert to integer: One=1, Two=2, Three=3, Four=4, Five=5)
- **Relative URL** of the book (from the `href` attribute inside the `h3 > a` tag)

Store the data in a pandas DataFrame and display the first 5 rows.

*Hint: Books are contained in `<article class="product_pod">` elements.*

In [4]:
url = 'http://books.toscrape.com/'
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

books = soup.find_all('article', class_='product_pod')
print(f"Found {len(books)} books on homepage\n")

book_data = []
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

for book in books:
    link = book.find('h3').find('a') if book.find('h3') else None
    title = link.get('title', '') if link else ''
    relative_url = link.get('href', '') if link else ''

    price_el = book.find('p', class_='price_color')
    price_text = price_el.get_text(strip=True) if price_el else ''
    price = float(price_text.replace('£', '')) if price_text else None

    avail_el = book.find('p', class_='instock availability')
    availability = avail_el.get_text(strip=True) if avail_el else ''

    rating_el = book.find('p', class_=re.compile('star-rating'))
    rating_class = rating_el.get('class', [])[-1] if rating_el else ''
    rating = rating_map.get(rating_class, 0)

    book_data.append({
        'title': title,
        'price': price,
        'availability': availability,
        'rating': rating,
        'relative_url': relative_url
    })

df_homepage = pd.DataFrame(book_data)
print("First 5 books from homepage:")
df_homepage.head()

Found 20 books on homepage

First 5 books from homepage:


,title,price,availability,rating,relative_url
0,A Light in the Attic,51.77,In stock,3,catalogue/a-light-in-the-attic_1000/index.html
1,Tipping the Velvet,53.74,In stock,1,catalogue/tipping-the-velvet_999/index.html
2,Soumission,50.10,In stock,1,catalogue/soumission_998/index.html
3,Sharp Objects,47.82,In stock,4,catalogue/sharp-objects_997/index.html
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,catalogue/sapiens-a-brief-history-of-humankind...


## Task 1.3: Save to CSV (5 points)

**Instructions:** Save your DataFrame to a CSV file named `books_homepage.csv`.

In [5]:
df_homepage.to_csv('books_homepage.csv', index=False)
print("Saved to books_homepage.csv")
print(f"File size: {len(df_homepage)} books")

Saved to books_homepage.csv
File size: 20 books


## Task 1.4: Extract Book Details from Individual Pages (5 points)

**Instructions:**

Choose any 3 books from your DataFrame. For each book:

1. Construct the full URL (use `urljoin` with the base URL)
2. Visit the book's detail page
3. Extract the **product description** (the text inside the meta tag with name="description")
4. Extract the **UPC** (from the table in the product detail page)

Print the results for each book.

*Hint: The description is in a `<meta name="description">` tag.*
*Hint: UPC is in the first `<tr>` of the product information table.*

In [6]:
base_url = 'http://books.toscrape.com/'
sample_books = df_homepage.head(3)

for idx, book in sample_books.iterrows():
    print(f"\nBook {idx+1}: {book['title']}")
    print("-" * 30)
    full_url = urljoin(base_url, book['relative_url'])
    try:
        resp = requests.get(full_url, timeout=10)
        resp.raise_for_status()
        page_soup = BeautifulSoup(resp.content, 'html.parser')
        meta_desc = page_soup.find('meta', attrs={'name': 'description'})
        description = meta_desc.get('content', '').strip() if meta_desc else ''
        table = page_soup.find('table', class_='table table-striped')
        upc = ''
        if table:
            rows = table.find_all('tr')
            for row in rows:
                th = row.find('th')
                if th and th.get_text(strip=True) == 'UPC':
                    td = row.find('td')
                    upc = td.get_text(strip=True) if td else ''
                    break
        print(f"Description: {description[:100]}..." if len(description) > 100 else f"Description: {description}")
        print(f"UPC: {upc}")
    except Exception as e:
        print(f"Error: {e}")
    time.sleep(random.uniform(1, 2))
print("\nTask 1.4 complete")


Book 1: A Light in the Attic
------------------------------
Description: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and...
UPC: a897fe39b1053632

Book 2: Tipping the Velvet
------------------------------
Description: "Erotic and absorbing...Written with starling power."--"The New York Times Book Review " Nan King, a...
UPC: 90fa61229261140a

Book 3: Soumission
------------------------------
Description: Dans une France assez proche de la nôtre, un homme s’engage dans la carrière universitaire. Peu moti...
UPC: 6957f44c3847a760

Task 1.4 complete


# Part 2: Pagination and Multiple Pages (30 points)

## Task 2.1: Implement Pagination Logic (15 points)

**Instructions:**

Write a function that scrapes all books from the first **3 pages** of the catalogue.

- Base URL: `http://books.toscrape.com/catalogue/page-1.html`
- Pages increment: page-1.html, page-2.html, page-3.html
- Extract the same fields as in Task 1.2
- Add a `page_number` column to track which page each book came from
- Implement polite scraping: wait 1-2 seconds between page requests

Return a DataFrame with all books from all pages.

In [7]:
# Your Task 2.1 code here

def scrape_multiple_pages(start_page=1, end_page=3):
    """
    Scrape books from multiple pages

    Args:
        start_page: First page number
        end_page: Last page number

    Returns:
        DataFrame with books from all pages
    """
    all_books = []
    base_url = 'http://books.toscrape.com/catalogue/page-{}.html'
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

    for page in range(start_page, end_page + 1):
        print(f"Scraping page {page}...")
        url = base_url.format(page)
        response = requests.get(url)

        if response.status_code != 200:
            print(f"  Failed to load page {page}")
            continue

        soup = BeautifulSoup(response.content, 'html.parser')
        books = soup.find_all('article', class_='product_pod')
        print(f"  Found {len(books)} books")

        for book in books:
            try:
                link = book.find('h3').find('a') if book.find('h3') else None
                title = link.get('title', '') if link else ''
                rel_url = link.get('href', '') if link else ''
                price_el = book.find('p', class_='price_color')
                pt = price_el.get_text(strip=True) if price_el else ''
                price = float(pt.replace('£', '')) if pt else None
                avail_el = book.find('p', class_='instock availability')
                availability = avail_el.get_text(strip=True) if avail_el else ''
                rating_el = book.find('p', class_=re.compile('star-rating'))
                rclass = rating_el.get('class', [])[-1] if rating_el else ''
                rating = rating_map.get(rclass, 0)
                all_books.append({'title': title, 'price': price, 'availability': availability,
                                 'rating': rating, 'relative_url': rel_url, 'page_number': page})
            except Exception as e:
                print(f"  Error parsing book: {e}")

        if page < end_page:
            delay = random.uniform(1, 2)
            print(f"  Waiting {delay:.2f} seconds...")
            time.sleep(delay)

    return pd.DataFrame(all_books)

# Test the function
df_multipage = scrape_multiple_pages(1, 3)
print(f"\nTotal books scraped: {len(df_multipage)}")
print(f"Books per page:\n{df_multipage['page_number'].value_counts().sort_index()}")

# Display sample
df_multipage.head(10)


Scraping page 1...
  Found 20 books
  Waiting 1.14 seconds...
Scraping page 2...
  Found 20 books
  Waiting 1.37 seconds...
Scraping page 3...
  Found 20 books

Total books scraped: 60
Books per page:
page_number
1    20
2    20
3    20
Name: count, dtype: int64


,title,price,availability,rating,relative_url,page_number
0,A Light in the Attic,51.77,In stock,3,a-light-in-the-attic_1000/index.html,1
1,Tipping the Velvet,53.74,In stock,1,tipping-the-velvet_999/index.html,1
2,Soumission,50.10,In stock,1,soumission_998/index.html,1
3,Sharp Objects,47.82,In stock,4,sharp-objects_997/index.html,1
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,sapiens-a-brief-history-of-humankind_996/index...,1
5,The Requiem Red,22.65,In stock,1,the-requiem-red_995/index.html,1
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4,the-dirty-little-secrets-of-getting-your-dream...,1
7,The Coming Woman: A Novel Based on the Life of...,17.93,In stock,3,the-coming-woman-a-novel-based-on-the-life-of-...,1
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,4,the-boys-in-the-boat-nine-americans-and-their-...,1
9,The Black Maria,52.15,In stock,1,the-black-maria_991/index.html,1


## Task 2.2: Error Handling and Robustness (10 points)

**Instructions:**

Modify your function to handle common errors:
- Use try-except for network errors
- Check if the page exists (status code 200)
- Handle missing elements gracefully (use conditional statements)
- Print progress updates
- Continue scraping even if one page fails

Create an enhanced version called `scrape_multiple_pages_robust()`.

In [8]:
# Your Task 2.2 code here

def scrape_multiple_pages_robust(start_page=1, end_page=5, max_retries=3):
    """
    Enhanced scraper with error handling and retries
    """
    all_books = []
    failed_pages = []
    base_url = 'http://books.toscrape.com/catalogue/page-{}.html'
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

    print(f"Starting scrape from page {start_page} to {end_page}")
    print("=" * 50)

    for page in range(start_page, end_page + 1):
        print(f"\nProcessing page {page}...")
        url = base_url.format(page)
        success = False

        for attempt in range(max_retries):
            try:
                response = requests.get(url, timeout=10)

                if response.status_code == 200:
                    success = True
                    break
                else:
                    print(f"  Attempt {attempt+1}: Status code {response.status_code}")

            except requests.exceptions.RequestException as e:
                print(f"  Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Waiting {wait} seconds before retry...")
                time.sleep(wait)

        if not success:
            print(f"  Failed to load page {page} after {max_retries} attempts")
            failed_pages.append(page)
            continue

        try:
            soup = BeautifulSoup(response.content, 'html.parser')
            books = soup.find_all('article', class_='product_pod')
            if not books:
                print(f"  Warning: No books found on page {page}")
            page_books = 0
            for book in books:
                try:
                    link = book.find('h3').find('a') if book.find('h3') else None
                    title = link.get('title', '') if link else ''
                    price_el = book.find('p', class_='price_color')
                    pt = price_el.get_text(strip=True) if price_el else ''
                    price = float(pt.replace('£', '')) if pt else None
                    avail_el = book.find('p', class_='instock availability')
                    availability = avail_el.get_text(strip=True) if avail_el else ''
                    rating_el = book.find('p', class_=re.compile('star-rating'))
                    rclass = rating_el.get('class', [])[-1] if rating_el else ''
                    rating = rating_map.get(rclass, 0)
                    rel_url = link.get('href', '') if link else ''
                    all_books.append({'title': title, 'price': price, 'availability': availability,
                                     'rating': rating, 'relative_url': rel_url, 'page_number': page})
                    page_books += 1
                except Exception as e:
                    print(f"  Error parsing individual book: {e}")
                    continue

            print(f"  Successfully scraped {page_books} books from page {page}")

        except Exception as e:
            print(f"  Error parsing page {page}: {e}")
            failed_pages.append(page)

        if page < end_page:
            time.sleep(random.uniform(1, 2))

    # Summary
    print("\n" + "=" * 50)
    print(f"Scraping complete!")
    print(f"Total books scraped: {len(all_books)}")
    if failed_pages:
        print(f"Failed pages: {failed_pages}")

    return pd.DataFrame(all_books)

# Test the robust version
df_robust = scrape_multiple_pages_robust(1, 5)
print(f"\nDataFrame shape: {df_robust.shape}")
df_robust.head()

Starting scrape from page 1 to 5

Processing page 1...
  Successfully scraped 20 books from page 1

Processing page 2...
  Successfully scraped 20 books from page 2

Processing page 3...
  Successfully scraped 20 books from page 3

Processing page 4...
  Successfully scraped 20 books from page 4

Processing page 5...
  Successfully scraped 20 books from page 5

Scraping complete!
Total books scraped: 100

DataFrame shape: (100, 6)


,title,price,availability,rating,relative_url,page_number
0,A Light in the Attic,51.77,In stock,3,a-light-in-the-attic_1000/index.html,1
1,Tipping the Velvet,53.74,In stock,1,tipping-the-velvet_999/index.html,1
2,Soumission,50.10,In stock,1,soumission_998/index.html,1
3,Sharp Objects,47.82,In stock,4,sharp-objects_997/index.html,1
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,sapiens-a-brief-history-of-humankind_996/index...,1


## Task 2.3: Save Combined Data (5 points)

**Instructions:** Save your scraped data from Task 2.2 to `books_multipage.csv`.

In [9]:
# Your Task 2.3 code here
df_robust.to_csv('books_multipage.csv', index=False)
print(f"Saved {len(df_robust)} books to books_multipage.csv")

# Quick verification
df_check = pd.read_csv('books_multipage.csv')
print(f"Verified: {len(df_check)} books loaded from CSV")

Saved 100 books to books_multipage.csv
Verified: 100 books loaded from CSV


# Part 3: Category-Based Scraping (25 points)

## Task 3.1: Extract Category Links (10 points)

**Instructions:**

Extract all book categories from the sidebar on the homepage:

1. Navigate to the homepage: `http://books.toscrape.com/`
2. Find the sidebar div with class "side_categories"
3. Extract all category names and their URLs
4. Exclude the "Books" category (which is just a header)
5. Store in a dictionary: `{category_name: category_url}`

Return a dictionary of all categories.

In [10]:
# Your Task 3.1 code here

def get_categories():
    """
    Extract all book categories from the sidebar

    Returns:
        Dictionary with category names and full URLs
    """
    url = 'http://books.toscrape.com/'
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    sidebar = soup.find('div', class_='side_categories')
    if not sidebar:
        return {}
    all_links = sidebar.find_all('a')
    category_links = all_links[1:]  # Skip first (Books)
    categories = {}
    for link in category_links:
        name = link.get_text(strip=True)
        relative_url = link.get('href', '')
        full_url = urljoin(url, relative_url)
        categories[name] = full_url
    return categories

# Get categories
categories = get_categories()
print(f"Found {len(categories)} categories:\n")
for name, url in list(categories.items())[:10]:  # Show first 10
    print(f"  {name}: {url}")

if len(categories) > 10:
    print(f"  ... and {len(categories) - 10} more")

Found 50 categories:

  Travel: http://books.toscrape.com/catalogue/category/books/travel_2/index.html
  Mystery: http://books.toscrape.com/catalogue/category/books/mystery_3/index.html
  Historical Fiction: http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
  Sequential Art: http://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
  Classics: http://books.toscrape.com/catalogue/category/books/classics_6/index.html
  Philosophy: http://books.toscrape.com/catalogue/category/books/philosophy_7/index.html
  Romance: http://books.toscrape.com/catalogue/category/books/romance_8/index.html
  Womens Fiction: http://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html
  Fiction: http://books.toscrape.com/catalogue/category/books/fiction_10/index.html
  Childrens: http://books.toscrape.com/catalogue/category/books/childrens_11/index.html
  ... and 40 more


## Task 3.2: Scrape a Complete Category (15 points)

**Instructions:**

Choose **one category** (e.g., "Mystery", "Travel", or "Science Fiction") and scrape **all books** in that category, including all paginated pages.

Requirements:
- Handle pagination within the category
- Extract the same fields as before (title, price, availability, rating, URL)
- Add a `category` column
- Implement polite delays
- Handle errors gracefully
- Save results to `books_category.csv`

Print summary statistics (total books, price range, average rating).

In [11]:
# Your Task 3.2 code here

def scrape_category(category_name):
    """
    Scrape all books from a specific category

    Args:
        category_name: Name of the category to scrape

    Returns:
        DataFrame with all books from the category
    """
    categories = get_categories()
    if category_name not in categories:
        print(f"Category '{category_name}' not found")
        print(f"Available categories: {list(categories.keys())}")
        return None
    base_category_url = categories[category_name]
    print(f"Scraping category: {category_name}")
    print(f"Base URL: {base_category_url}")

    all_books = []
    page_num = 1
    has_next_page = True

    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

    while has_next_page:
        if page_num == 1:
            page_url = base_category_url
        else:
            if base_category_url.endswith('index.html'):
                page_url = base_category_url.replace('index.html', f'page-{page_num}.html')
            else:
                page_url = base_category_url.rstrip('/') + f'/page-{page_num}.html'

        print(f"  Scraping page {page_num}: {page_url}")

        try:
            response = requests.get(page_url, timeout=10)
            if response.status_code != 200:
                print(f"  Page {page_num} not found (status {response.status_code})")
                break
            soup = BeautifulSoup(response.content, 'html.parser')
            books = soup.find_all('article', class_='product_pod')

            if not books:
                print(f"  No books found on page {page_num}")
                break

            for book in books:
                try:
                    link = book.find('h3').find('a') if book.find('h3') else None
                    title = link.get('title', '') if link else ''
                    price_el = book.find('p', class_='price_color')
                    pt = price_el.get_text(strip=True) if price_el else ''
                    price = float(pt.replace('£', '')) if pt else None
                    avail_el = book.find('p', class_='instock availability')
                    availability = avail_el.get_text(strip=True) if avail_el else ''
                    rating_el = book.find('p', class_=re.compile('star-rating'))
                    rclass = rating_el.get('class', [])[-1] if rating_el else ''
                    rating = rating_map.get(rclass, 0)
                    relative_url = link.get('href', '') if link else ''
                    all_books.append({
                        'title': title,
                        'price': price,
                        'availability': availability,
                        'rating': rating,
                        'relative_url': relative_url,
                        'category': category_name,
                        'page_in_category': page_num
                    })


                except Exception as e:
                    print(f"    Error parsing book: {e}")

            print(f"    Found {len(books)} books on page {page_num}")

            next_button = soup.find('li', class_='next')
            has_next_page = next_button is not None

            if has_next_page:
                page_num += 1
                delay = random.uniform(1.5, 3)
                print(f"    Waiting {delay:.2f} seconds before next page...")
                time.sleep(delay)

        except requests.exceptions.RequestException as e:
            print(f"  Error fetching page {page_num}: {e}")
            break

    df = pd.DataFrame(all_books)

    print("\n" + "=" * 50)
    print(f"Category scrape complete: {category_name}")
    print(f"Total books scraped: {len(df)}")
    print(f"Pages scraped: {page_num}")

    if len(df) > 0:
        print(f"Price range: {df['price'].min():.2f} - {df['price'].max():.2f}")
        print(f"Average rating: {df['rating'].mean():.2f}")

    return df

# Choose a category to scrape (e.g. Mystery, Travel, Science Fiction)
category_to_scrape = 'Travel'

df_category = scrape_category(category_to_scrape)

# Save to CSV
if df_category is not None and len(df_category) > 0:
    df_category.to_csv('books_category.csv', index=False)
    print(f"\nSaved to books_category.csv")

    # Display sample
    print("\nSample of scraped books:")
    df_category.head(10)

Scraping category: Travel
Base URL: http://books.toscrape.com/catalogue/category/books/travel_2/index.html
  Scraping page 1: http://books.toscrape.com/catalogue/category/books/travel_2/index.html
    Found 11 books on page 1

Category scrape complete: Travel
Total books scraped: 11
Pages scraped: 1
Price range: 23.21 - 56.88
Average rating: 2.73

Saved to books_category.csv

Sample of scraped books:



# Part 4: Data Aggregation and Summary (10 points)

**Instructions:**

Load all three CSV files you've created and combine them into a single DataFrame:
- `books_homepage.csv`
- `books_multipage.csv`
- `books_category.csv`

Add a `source` column to identify where each record came from. Display the combined DataFrame info and shape.

In [13]:
# Your Task 4 code here

import os

csv_files = ['books_homepage.csv', 'books_multipage.csv', 'books_category.csv']
dataframes = []

for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)

        # Add source column (remove .csv extension)
        source_name = file.replace('.csv', '')
        df['source'] = source_name

        dataframes.append(df)
        print(f"Loaded {file}: {len(df)} rows")
    else:
        print(f"Warning: {file} not found")

# Combine all dataframes
if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True)
    print("\n" + "=" * 50)
    print("Shape:", combined_df.shape)
    print("Length:", len(combined_df))
    print("Columns:", list(combined_df.columns))
    print("\nRecords by source:")
    print(combined_df['source'].value_counts())
    print("\nFirst few rows:")
    print(combined_df.head())
else:
    print("No data files found. Please complete previous tasks first.")

Loaded books_homepage.csv: 20 rows
Loaded books_multipage.csv: 100 rows
Loaded books_category.csv: 11 rows

Shape: (131, 9)
Length: 131
Columns: ['title', 'price', 'availability', 'rating', 'relative_url', 'source', 'page_number', 'category', 'page_in_category']

Records by source:
source
books_multipage    100
books_homepage      20
books_category      11
Name: count, dtype: int64

First few rows:
                                   title  price availability  rating                                       relative_url          source  page_number category  page_in_category
0                   A Light in the Attic  51.77     In stock       3     catalogue/a-light-in-the-attic_1000/index.html  books_homepage          NaN      NaN               NaN
1                     Tipping the Velvet  53.74     In stock       1        catalogue/tipping-the-velvet_999/index.html  books_homepage          NaN      NaN               NaN
2                             Soumission  50.10     In stock       1  

# Part 5: Reflection and Submission (5 points)

## Reflection Questions

Answer the following questions in the markdown cell below:

1. What challenges did you encounter while scraping, and how did you solve them?

2. Why is polite scraping (delays between requests) important?

3. How would you modify your scraper if the website had dynamic content loaded with JavaScript?

4. What ethical considerations should you keep in mind when scraping websites?

5. If you were to scrape 1000+ pages from this site, what improvements would you make to your code?

**Your Reflection:**

1. Some pages had missing elements or different structure, so I used conditional checks (e.g. if link else '') and try/except so one bad book or page does not stop the whole run. For categories, the URL pattern for page 2+ was different from the first page.

2. Delays between requests avoid overloading the server and reduce the chance of being blocked. It also makes the scraper behave more like a normal user.

3. If the site loaded content with JavaScript, BeautifulSoup would only see the initial HTML. I would use something like Selenium or Playwright to drive a browser and wait for the content to load, then parse the resulting HTML.

4. Check the site's terms of use and robots.txt, respect rate limits, do not scrape personal data without consent, and use the data only for the stated purpose (e.g. assignment).

5. I would add caching (e.g. save raw HTML and skip already-fetched pages), run in batches with longer delays, maybe use async requests for speed while staying polite, and log progress to a file so I can resume if it stops.

---
## Submission Instructions

Before submitting, ensure you have:

- [ ] Run all cells and verified they execute without errors
- [ ] Saved all CSV files: `books_homepage.csv`, `books_multipage.csv`, `books_category.csv`
- [ ] Answered the reflection questions
- [ ] Downloaded your notebook from Colab (File -> Download -> Download .ipynb)

**Submit both:**
1. Your Jupyter Notebook file (.ipynb)
2. All generated CSV files

---
**End of Assignment**